# Full-throttle speed loss in 2026 F1 qualifying: season run

Runs the Japan-validated pipeline (`flatout.run_session`) on the main Qualifying session of every 2026 event held so far (conventional and sprint weekends; Sprint Qualifying is not used). Each session's tables go to `data/season/`; the QC summary has one row per session. Validation 1 needs each circuit's official length and is replaced here by a check that needs none: per push lap, car-data distance vs X/Y distance. OpenF1 brake values come from `openf1_session_key`; without a key, brake is taken at face value and logged. Sessions are tagged pre/post-Miami by Qualifying date relative to the Miami GP Qualifying.

**Calendar note.** The 2026 Bahrain and Saudi Arabian GPs were cancelled (OpenF1 `is_cancelled` for Sakhir and Jeddah); the Bahrain GP was rescheduled to Sepang, Malaysia (OpenF1 session 11730, location "Kuala Lumpur"), which FastF1 correctly lists as "Bahrain Grand Prix".

In [1]:
import re
import time
from pathlib import Path

import fastf1
import numpy as np
import pandas as pd

from flatout import distance_agreement, openf1_session_key, run_session, to_seconds

YEAR = 2026
DIST_FLAG_PCT = 3  # push laps whose car-data and X/Y distances differ by more than this are flagged
MIN_CLEAN_DRIVERS = 10  # reporting rule: a straight is reported for a session only with >= 10 clean drivers
OUT = Path("data/season")
OUT.mkdir(parents=True, exist_ok=True)

schedule = fastf1.get_event_schedule(YEAR, include_testing=False)
events = schedule[schedule.EventFormat.isin(["conventional", "sprint_qualifying"])].copy()  # all have a main "Qualifying"
events["QualiDateUtc"] = [next(r[f"Session{i}DateUtc"] for i in range(1, 6) if r[f"Session{i}"] == "Qualifying")
                          for _, r in events.iterrows()]
miami_q = events.loc[events.EventName.str.contains("Miami"), "QualiDateUtc"].iloc[0]  # rule changes apply from Miami onward
events["Period"] = np.where(events.QualiDateUtc < miami_q, "pre_miami", "post_miami")
now = pd.Timestamp.now(tz="UTC").tz_localize(None)
events = events[events.QualiDateUtc < now]
print(f"Miami GP qualifying (UTC): {miami_q}; now (UTC): {now:%Y-%m-%d %H:%M}; sessions held so far: {len(events)}\n")
print(events[["RoundNumber", "EventName", "Location", "QualiDateUtc", "Period"]].to_string(index=False))

req         WARNING 	DEFAULT CACHE ENABLED! (36.0 KB) C:\Users\User\AppData\Local\Temp\fastf1


Miami GP qualifying (UTC): 2026-05-02 20:00:00; now (UTC): 2026-10-14 09:58; sessions held so far: 17

 RoundNumber             EventName          Location        QualiDateUtc     Period
           1 Australian Grand Prix         Melbourne 2026-03-07 05:00:00  pre_miami
           2    Chinese Grand Prix          Shanghai 2026-03-14 07:00:00  pre_miami
           3   Japanese Grand Prix            Suzuka 2026-03-28 06:00:00  pre_miami
           4      Miami Grand Prix     Miami Gardens 2026-05-02 20:00:00 post_miami
           5   Canadian Grand Prix          Montréal 2026-05-23 20:00:00 post_miami
           6     Monaco Grand Prix       Monte Carlo 2026-06-06 14:00:00 post_miami
           7  Barcelona Grand Prix         Barcelona 2026-06-13 14:00:00 post_miami
           8   Austrian Grand Prix         Spielberg 2026-06-27 14:00:00 post_miami
           9    British Grand Prix       Silverstone 2026-07-04 15:00:00 post_miami
          10    Belgian Grand Prix Spa-Francorchamps 2026

## Run (one session at a time; a failure is logged and the loop continues)

In [2]:
rows = []
t_start = time.perf_counter()
for ev in events.itertuples():
    row = dict(Round=ev.RoundNumber, Event=ev.EventName, Period=ev.Period)
    t0 = time.perf_counter()
    try:
        tables, qc = run_session(YEAR, ev.RoundNumber, openf1_session=openf1_session_key(YEAR, ev.RoundNumber))
        dist = distance_agreement(tables["push"])
        stem = OUT / f"{ev.RoundNumber:02d}_{re.sub(r'[^A-Za-z0-9]+', '_', ev.Location).strip('_').lower()}"
        to_seconds(tables["sectors"]).to_csv(f"{stem}_sectors.csv", index=False)
        to_seconds(tables["segments"]).to_csv(f"{stem}_segments.csv", index=False)
        tables["exclusions"].to_csv(f"{stem}_exclusions.csv", index=False)
        dist.to_csv(f"{stem}_distance_qc.csv", index=False)
        cov = tables["coverage"]
        part = lambda k: cov.loc[k].to_dict() if k in cov.index.get_level_values(0) else {}
        row.update(Status="ok", **qc, Exclusions=len(tables["exclusions"]),
                   CleanPerStraight=part("straight"), CleanPerSector=part("sector"),
                   DistDiff_median_pct=dist.Diff_pct.abs().median(), DistDiff_max_pct=dist.Diff_pct.abs().max(),
                   LapsOver3pct=int((dist.Diff_pct.abs() > DIST_FLAG_PCT).sum()))
    except Exception as e:
        row.update(Status=f"error: {type(e).__name__}: {e}")
    row["Runtime_s"] = time.perf_counter() - t0
    rows.append(row)
    print(f"round {ev.RoundNumber:2d} {ev.EventName:<26} {row['Status'][:60]:<60} {row['Runtime_s']:6.0f} s", flush=True)
season_qc = pd.DataFrame(rows)
season_qc.to_csv(OUT / "qc_summary.csv", index=False)
print(f"\nTotal runtime: {(time.perf_counter() - t_start) / 60:.1f} min")

core        WARNING 	No lap data for driver 18


core        WARNING 	No lap data for driver 55


core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)


core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 55)


round  1 Australian Grand Prix      ok                                                               35 s


...

round 15 Azerbaijan Grand Prix      ok                                                               20 s


data        WARNING 	Failed to load circuit info


round 16 Bahrain Grand Prix         ok                                                               10 s


round 17 Singapore Grand Prix       ok                                                               16 s



Total runtime: 4.6 min


## QC summary (one row per session)

In [3]:
ok = season_qc.Status.eq("ok")
season_qc["Reportable"] = [sorted(k for k, n in d.items() if n >= MIN_CLEAN_DRIVERS) if isinstance(d, dict) else [] for d in season_qc.CleanPerStraight]
season_qc["NReportable"] = season_qc.Reportable.str.len().where(ok)
season_qc.to_csv(OUT / "qc_summary.csv", index=False)
cols = ["Round", "Event", "Period", "Status", "brake_source", "circuit_info", "push_laps", "push_lap_frozen_pct_mean",
        "DistDiff_median_pct", "DistDiff_max_pct", "LapsOver3pct", "CleanPerStraight", "NReportable", "Reportable", "Exclusions", "Runtime_s"]
pd.set_option("display.max_colwidth", 90)
season_qc.reindex(columns=cols).rename(columns={"push_lap_frozen_pct_mean": "FreezeShare_pct"}).round(1)

,Round,Event,Period,Status,brake_source,circuit_info,push_laps,FreezeShare_pct,DistDiff_median_pct,DistDiff_max_pct,LapsOver3pct,CleanPerStraight,NReportable,Reportable,Exclusions,Runtime_s
0,1,Australian Grand Prix,pre_miami,ok,OpenF1,ok,102,0.0,0.4,3.3,1,"{'T1': 2, 'T11': 18, 'T3': 5, 'T6': 7, 'T9': 19}",2,"[T11, T9]",248,35.2
1,2,Chinese Grand Prix,pre_miami,ok,OpenF1,ok,102,10.3,1.2,55.5,22,"{'T1': 18, 'T14': 21, 'T6': 1, 'T8': 19}",3,"[T1, T14, T8]",469,16.3
2,3,Japanese Grand Prix,pre_miami,ok,OpenF1,ok,108,10.6,0.5,39.8,10,"{'T1': 19, 'T13': 22, 'T16': 21}",3,"[T1, T13, T16]",580,14.7
3,4,Miami Grand Prix,post_miami,ok,OpenF1,ok,100,0.2,0.4,62.6,3,"{'T1': 16, 'T11': 22, 'T17': 22, 'T4': 6, 'T5': 4}",3,"[T1, T11, T17]",158,14.4
4,5,Canadian Grand Prix,post_miami,ok,OpenF1,ok,120,0.1,0.3,3.4,1,"{'T1': 14, 'T13': 22, 'T8': 14}",3,"[T1, T13, T8]",328,17.2
5,6,Monaco Grand Prix,post_miami,ok,OpenF1,ok,132,0.0,0.3,2.0,0,{'T10': 11},1,[T10],403,17.6
6,7,Barcelona Grand Prix,post_miami,ok,OpenF1,ok,74,0.1,0.3,1.4,0,{'T1': 22},1,[T1],155,12.9
7,8,Austrian Grand Prix,post_miami,ok,OpenF1,ok,91,0.2,0.2,3.2,1,"{'T1': 22, 'T3': 22, 'T4': 22, 'T9': 22}",4,"[T1, T3, T4, T9]",97,13.9
8,9,British Grand Prix,post_miami,ok,OpenF1,ok,92,0.0,0.2,0.9,0,"{'T1': 22, 'T10': 1, 'T11': 3, 'T12': 22, 'T13': 1, 'T15': 22, 'T6': 22, 'T9': 21}",5,"[T1, T12, T15, T6, T9]",180,15.6
9,10,Belgian Grand Prix,post_miami,ok,OpenF1,ok,87,0.2,0.1,0.8,0,"{'T12': 22, 'T18': 22, 'T5': 22}",3,"[T12, T18, T5]",188,14.8


## Coverage per period (sessions that ran)

Freeze share and clean-driver coverage, pre vs post Miami. Straight coverage counts drivers with a clean segment per labelled straight (main straight included); a straight is reportable with >= `MIN_CLEAN_DRIVERS` clean drivers.

In [4]:
done = season_qc[ok]
straight_n = lambda s: [n for d in s for n in d.values()]
by_period = done.groupby("Period", sort=False).agg(
    Sessions=("Round", "size"), PushLaps=("push_laps", "sum"),
    FreezeShare_mean_pct=("push_lap_frozen_pct_mean", "mean"), FreezeShare_max_pct=("push_lap_frozen_pct_mean", "max"),
    Straights=("CleanPerStraight", lambda s: len(straight_n(s))),
    ReportableStraights=("NReportable", "sum"),
    CleanDriversPerStraight_median=("CleanPerStraight", lambda s: np.median(straight_n(s))),
    CleanDriversPerSector_median=("CleanPerSector", lambda s: np.median(straight_n(s))),
    SessionsWithoutCircuitInfo=("circuit_info", lambda s: int((s != "ok").sum())),
)
by_period.round(1)

,Sessions,PushLaps,FreezeShare_mean_pct,FreezeShare_max_pct,Straights,ReportableStraights,CleanDriversPerStraight_median,CleanDriversPerSector_median,SessionsWithoutCircuitInfo
Period,,,,,,,,,
pre_miami,3,312,7.0,10.6,12,8,18.5,19.0,0
post_miami,14,1373,0.2,1.4,42,33,22.0,22.0,2
